# 17. Tensors & Tensor Operations (Einsum)

Welcome to **Tensors and Tensor Operations**!

In modern Artificial Intelligence and Deep Learning, data is rarely just a 1D vector or a 2D matrix. High-dimensional data—such as image batches, video clips, sentence embeddings, and multi-head attention weights in Transformers—are naturally represented as **Tensors**.

---

## 1. What is a Tensor?

A **tensor** is a mathematical object that generalizes scalars, vectors, and matrices to higher dimensions:
- **0D Tensor (Rank 0)**: **Scalar** (a single number, e.g., $x = 5$)
- **1D Tensor (Rank 1)**: **Vector** (a 1D array of numbers, e.g., $\mathbf{x} = [1, 2, 3]$)
- **2D Tensor (Rank 2)**: **Matrix** (a 2D grid of numbers, e.g., $\mathbf{A} \in \mathbb{R}^{m 	imes n}$)
- **3D Tensor (Rank 3)**: A cube of numbers (e.g., RGB Image of shape `(Height, Width, Channels)` or a single sentence of shape `(Sequence_Length, Embedding_Dim)`)
- **4D Tensor (Rank 4)**: A batch of images of shape `(Batch_Size, Channels, Height, Width)` or multi-head attention `(Batch_Size, Num_Heads, Seq_Len, Head_Dim)`)
- **nD Tensor (Rank $n$)**: An $n$-dimensional array of numbers.

### Key Terminology:
1. **Rank / Order / Dimension count**: The number of axes/indices needed to address an element.
2. **Shape**: The size along each axis, e.g., `(32, 8, 128, 64)`.
3. **Strides**: The step size in memory to jump from one element to the next along an axis.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 0D, 1D, 2D, 3D, 4D Tensors in NumPy
scalar = np.array(42)
vector = np.array([1.0, 2.5, 3.8])
matrix = np.array([[1, 2, 3], [4, 5, 6]])
tensor_3d = np.random.randn(2, 3, 4) # e.g., 2 batches of 3x4 feature maps
tensor_4d = np.random.randn(8, 3, 32, 32) # e.g., 8 RGB images of size 32x32

print(f"Scalar: Rank={scalar.ndim}, Shape={scalar.shape}")
print(f"Vector: Rank={vector.ndim}, Shape={vector.shape}")
print(f"Matrix: Rank={matrix.ndim}, Shape={matrix.shape}")
print(f"3D Tensor: Rank={tensor_3d.ndim}, Shape={tensor_3d.shape}")
print(f"4D Tensor: Rank={tensor_4d.ndim}, Shape={tensor_4d.shape}")


---

## 2. Fundamental Tensor Operations

### 2.1 Reshaping, Squeezing, and Unsqueezing
- **Reshape**: Changing the dimensions without changing the underlying data in memory.
- **Transpose / Permute**: Swapping dimensions (e.g., converting PyTorch `(Batch, Channels, H, W)` to Matplotlib `(H, W, Channels)`).
- **Broadcasting**: Automatically expanding dimensions of size 1 so shapes match for element-wise operations.


In [ ]:
# Visualizing Tensor Transposition and Permutation
# Simulate a batch of 2 color images (Batch, Channels, Height, Width)
batch_images = np.random.rand(2, 3, 64, 64)
print(f"Original shape (B, C, H, W): {batch_images.shape}")

# Convert to (Batch, Height, Width, Channels) for visualization/plotting
plottable_images = np.transpose(batch_images, (0, 2, 3, 1))
print(f"Permuted shape (B, H, W, C): {plottable_images.shape}")

# Squeezing and Unsqueezing (Expanding dims)
vec = np.array([1, 2, 3]) # shape (3,)
expanded = np.expand_dims(vec, axis=0) # shape (1, 3)
expanded_col = np.expand_dims(vec, axis=1) # shape (3, 1)
print(f"Expanded row vector: {expanded.shape}, Column vector: {expanded_col.shape}")


---

## 3. Batch Matrix Multiplication (BMM)

In Deep Learning, we almost never multiply single matrices $\mathbf{A} \mathbf{B}$. Instead, we multiply **batches** of matrices simultaneously:
$$\mathbf{C}_b = \mathbf{A}_b \mathbf{B}_b \quad 	ext{for } b = 1, 2, \dots, B$$
If $\mathbf{A} \in \mathbb{R}^{B 	imes M 	imes K}$ and $\mathbf{B} \in \mathbb{R}^{B 	imes K 	imes N}$, then $\mathbf{C} \in \mathbb{R}^{B 	imes M 	imes N}$.


In [ ]:
# Batch Matrix Multiplication in NumPy
batch_size = 4
M, K, N = 3, 5, 2

A = np.random.randn(batch_size, M, K)
B = np.random.randn(batch_size, K, N)

# Using np.matmul / @ operator (automatically handles batch dimensions)
C = A @ B

print("Batch A shape:", A.shape)
print("Batch B shape:", B.shape)
print("Batch C shape (A @ B):", C.shape)
# Verify with loop
assert np.allclose(C[0], A[0] @ B[0])
print("Batch multiplication verified successfully!")


---

## 4. Einstein Summation Notation (`einsum`)

### Why `einsum`?
Einstein summation convention is the **universal language of tensor contractions**. It provides a compact, elegant, and highly optimized syntax for expressing virtually any multidimensional tensor operation.

### The Syntax Rule:
`output = np.einsum('subscripts', *operands)`
- Indices appearing in multiple input terms but **not** in the output term are **summed over** (contracted).
- Indices appearing in both input and output are **retained**.

### Common Operations Table:

| Operation | Standard Math | Einsum Subscript |
| :--- | :--- | :--- |
| **Vector Dot Product** | $\mathbf{u} \cdot \mathbf{v} = \sum_i u_i v_i$ | `'i,i->'` |
| **Outer Product** | $\mathbf{u} \mathbf{v}^T$ ($A_{ij} = u_i v_j$) | `'i,j->ij'` |
| **Matrix-Vector Multiplication** | $\mathbf{y} = \mathbf{A} \mathbf{x}$ ($y_i = \sum_j A_{ij} x_j$) | `'ij,j->i'` |
| **Matrix-Matrix Multiplication** | $\mathbf{C} = \mathbf{A} \mathbf{B}$ ($C_{ik} = \sum_j A_{ij} B_{jk}$) | `'ij,jk->ik'` |
| **Matrix Trace** | $	ext{Tr}(\mathbf{A}) = \sum_i A_{ii}$ | `'ii->'` |
| **Matrix Transpose** | $\mathbf{A}^T$ | `'ij->ji'` |
| **Batch Matrix Multiplication** | $\mathbf{C}_b = \mathbf{A}_b \mathbf{B}_b$ | `'bij,bjk->bik'` |
| **Tensor Contraction (3D & 2D)** | $C_{b,j} = \sum_k A_{b,k} B_{k,j}$ | `'bk,kj->bj'` |


In [ ]:
# Demonstrating standard Einsum operations
u = np.array([1., 2., 3.])
v = np.array([4., 5., 6.])
M1 = np.array([[1., 2.], [3., 4.]])
M2 = np.array([[5., 6.], [7., 8.]])

# 1. Dot Product
dot_prod = np.einsum('i,i->', u, v)
print("1. Dot product (u . v):", dot_prod)

# 2. Outer Product
outer_prod = np.einsum('i,j->ij', u, v)
print("2. Outer product shape:", outer_prod.shape)

# 3. Matrix Multiplication
matmul_res = np.einsum('ij,jk->ik', M1, M2)
print("3. Matrix Multiplication:\n", matmul_res)

# 4. Matrix Trace
trace_res = np.einsum('ii->', M1)
print("4. Trace of M1 (1 + 4):", trace_res)

# 5. Batch Matrix Multiplication
A_batch = np.random.randn(8, 4, 5)
B_batch = np.random.randn(8, 5, 3)
C_batch = np.einsum('bij,bjk->bik', A_batch, B_batch)
print("5. Batch Matmul shape:", C_batch.shape)


---

## 5. Modern AI Application: Multi-Head Self-Attention via `einsum`

In modern Large Language Models (LLMs) and Vision Transformers (ViT), the self-attention formula is:
$$	ext{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = 	ext{Softmax}\left(rac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}}ight) \mathbf{V}$$

In a multi-head setting, the tensors have 4 dimensions:
- $\mathbf{Q} \in \mathbb{R}^{	ext{Batch} 	imes 	ext{Heads} 	imes 	ext{SeqLen}_q 	imes d_k}$
- $\mathbf{K} \in \mathbb{R}^{	ext{Batch} 	imes 	ext{Heads} 	imes 	ext{SeqLen}_k 	imes d_k}$
- $\mathbf{V} \in \mathbb{R}^{	ext{Batch} 	imes 	ext{Heads} 	imes 	ext{SeqLen}_k 	imes d_v}$

Without `einsum`, you would have to perform complicated transpositions and reshape operations. With `einsum`, it is written cleanly in two lines!


In [ ]:
def multi_head_attention_einsum(Q, K, V):
    """
    Compute Multi-Head Attention using Einsum.
    Shapes:
        Q: (batch, heads, seq_len_q, d_k)
        K: (batch, heads, seq_len_k, d_k)
        V: (batch, heads, seq_len_k, d_v)
    Returns:
        output: (batch, heads, seq_len_q, d_v)
        attn_weights: (batch, heads, seq_len_q, seq_len_k)
    """
    d_k = Q.shape[-1]
    
    # Step 1: Compute attention scores S = Q K^T / sqrt(d_k)
    # Subscripts: 'bhqd,bhkd->bhqk' (contracts dimension d_k)
    scores = np.einsum('bhqd,bhkd->bhqk', Q, K) / np.sqrt(d_k)
    
    # Step 2: Softmax over the last axis (seq_len_k)
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attn_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    
    # Step 3: Compute weighted sum of values: Output = Attn_Weights * V
    # Subscripts: 'bhqk,bhkd->bhqd' (contracts dimension seq_len_k)
    output = np.einsum('bhqk,bhkd->bhqd', attn_weights, V)
    
    return output, attn_weights

# Test with dummy inputs
batch_size, num_heads, seq_len, d_k = 2, 4, 8, 16
Q = np.random.randn(batch_size, num_heads, seq_len, d_k)
K = np.random.randn(batch_size, num_heads, seq_len, d_k)
V = np.random.randn(batch_size, num_heads, seq_len, d_k)

out, weights = multi_head_attention_einsum(Q, K, V)
print("Output tensor shape:", out.shape)
print("Attention weights shape:", weights.shape)
print("Sum of attention weights along row (should be 1.0):", weights[0, 0, 0].sum())


---

## 6. Summary & Key Takeaways

1. **Tensors** generalize scalar, vector, and matrix representations to $n$ dimensions.
2. High-dimensional tensor operations like **Broadcasting**, **Batch Matrix Multiplication (BMM)**, and **Transpositions** are fundamental to GPU computing.
3. **`einsum`** eliminates error-prone reshaping and provides expressive, readable, and highly optimized tensor algebra for attention mechanisms and neural network layers.
